# Магістерська робота: Дослідження та оцінка ризиків захворювань
## Ноутбук 1: Розвідувальний аналіз даних (EDA) та клінічний відбір ознак (Feature Selection)

* **Тема:** «Інтелектуальна система оцінки ризиків захворювань на основі методів пояснюваного штучного інтелекту»
* **Спеціальність:** 122 «Комп’ютерні науки»
* **Джерело даних:** U.S. Centers for Disease Control and Prevention (CDC) — BRFSS (253 680 записів).

---
### Мета дослідження:
1. Завантаження та перевірка цілісності офіційного набору даних CDC BRFSS.
2. **Науковий відбір ознак (Feature Selection):**
   * Збереження клінічних, біологічних та поведінкових маркерів здоров'я.
   * **Вилучення неетичного соціально-економічного шуму** (`Income`, `Education`, `AnyHealthcare`, `NoDocbcCost`) для запобігання соціальної упередженості моделі (Algorithmic Bias).
3. Аналіз взаємозв'язку між способом життя (куріння, дієта, активність) та розвитком серцево-судинних хвороб.
4. Збереження чистого навчального та тестового наборів даних у надійному форматі `.csv`.


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 11
plt.rcParams['figure.figsize'] = (10, 6)

print("Бібліотеки успішно завантажено!")


In [2]:
# Завантаження датасету
data_path = os.path.join('..', 'Datasets', 'heart_disease_health_indicators_BRFSS2015.csv')
if not os.path.exists(data_path):
    data_path = os.path.join('Datasets', 'heart_disease_health_indicators_BRFSS2015.csv')

df = pd.read_csv(data_path)
print(f"Початкова розмірність датасету: {df.shape[0]:,} рядків, {df.shape[1]} стовпців")


### 1. Клінічний розподіл та фільтрація ознак
Розділимо ознаки на домени відповідно до рекомендацій Європейського товариства кардіологів (ESC) та Американської асоціації серця (AHA).


In [3]:
# Ознаки, що вилучаються через соціально-економічний шум / відсутність прямого клінічного сенсу
noise_cols = ['Income', 'Education', 'AnyHealthcare', 'NoDocbcCost', 'CholCheck']

df_clean = df.drop(columns=noise_cols, errors='ignore')

print("Вилучено соціологічні ознаки:", noise_cols)
print(f"Залишено чистих медико-поведінкових ознак: {df_clean.shape[1] - 1}")
print("\nФінальний перелік клінічних факторів:")
for idx, col in enumerate(df_clean.drop(columns=['HeartDiseaseorAttack']).columns, 1):
    print(f"{idx:2d}. {col}")


### 2. Розподіл цільової патології та факторів способу життя

In [4]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Серцево-судинні захворювання
sns.countplot(x='HeartDiseaseorAttack', data=df_clean, ax=axes[0], palette=['#48bb78', '#f56565'])
axes[0].set_title('Серцево-судинні захворювання (ССЗ)', fontsize=12, fontweight='bold')
axes[0].set_xticklabels(['Здоровий (0)', 'Хвороба серця (1)'])
axes[0].set_xlabel('')

# 2. Цукровий діабет
sns.countplot(x='Diabetes', data=df_clean, ax=axes[1], palette=['#48bb78', '#ecc94b', '#e53e3e'])
axes[1].set_title('Цукровий діабет (0-ні, 1-предіабет, 2-діабет)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('')

# 3. Гіпертонія
sns.countplot(x='HighBP', data=df_clean, ax=axes[2], palette=['#48bb78', '#ed8936'])
axes[2].set_title('Гіпертонія (High Blood Pressure)', fontsize=12, fontweight='bold')
axes[2].set_xticklabels(['Норма (0)', 'Високий тиск (1)'])
axes[2].set_xlabel('')

plt.tight_layout()
plt.show()

print("Частка хворих на ССЗ:", round(df_clean['HeartDiseaseorAttack'].mean() * 100, 2), "%")


### 3. Кореляційний аналіз клінічних факторів

In [5]:
correlations = df_clean.corr()['HeartDiseaseorAttack'].sort_values(ascending=False)

plt.figure(figsize=(10, 7))
sns.barplot(x=correlations.values[1:], y=correlations.index[1:], palette='coolwarm')
plt.title('Кореляція клінічних факторів із хворобами серця (CDC BRFSS)', fontsize=13, fontweight='bold')
plt.xlabel('Коефіцієнт кореляції Пірсона')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

print("Топ-5 найсильніших клінічних факторів ризику:")
print(correlations.iloc[1:6])


### 4. Збереження чистого навчального та тестового наборів (CSV)

In [6]:
from sklearn.model_selection import train_test_split

X = df_clean.drop(columns=['HeartDiseaseorAttack'])
y = df_clean['HeartDiseaseorAttack'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Розмір навчальної вибірки: {X_train.shape[0]:,} пацієнтів")
print(f"Розмір тестової вибірки:   {X_test.shape[0]:,} пацієнтів")

processed_dir = os.path.join('..', 'Datasets', 'processed')
os.makedirs(processed_dir, exist_ok=True)

X_train.to_csv(os.path.join(processed_dir, 'X_train.csv'), index=False)
X_test.to_csv(os.path.join(processed_dir, 'X_test.csv'), index=False)
y_train.to_frame().to_csv(os.path.join(processed_dir, 'y_train.csv'), index=False)
y_test.to_frame().to_csv(os.path.join(processed_dir, 'y_test.csv'), index=False)

print(f"Чисті клінічні вибірки успішно збережено у: {processed_dir}")
